# Rover VLA — training on Colab

Trains the rover waypoint policy on a Colab GPU. Pick a **MODE** below.

| mode | what it does | needs |
|---|---|---|
| `A` | **stage-1 baseline** — frozen VLM backbone, action expert only. The stable recipe. | dataset |
| `B` | **constrained vision adaptation** — warm-start a checkpoint, unfreeze only the top-K vision layers (LM frozen). Tests whether adapting vision fixes colour grounding. | dataset + checkpoint |
| `C` | **deeper LM** — raise `num_vlm_layers` (SmolVLA truncates the LM to 16 of 32 by default). Tests whether the truncation is what blocks fine colour binding. Trains from base. | dataset |

Pipeline: **data (local Gazebo) → train (Colab) → eval (local sim + policy_server)**.
Colab does training only.

Upload from `rover/colab_upload/` to Drive `MyDrive/rover_vla/` first:
`rover_vla_v3.tar.gz` (always) and `stage1_v3_pretrained.tar.gz` (mode B only).

## 1. Config — set this, then Runtime ▸ Run all

In [ ]:
MODE = 'C'          # 'A' baseline | 'B' vision adaptation | 'C' deeper LM
STEPS = 10000
SAVE_FREQ = 500     # frequent: a dropped session loses <=1 checkpoint
VISION_TOPK = 2     # mode B: how many top vision layers to unfreeze
NUM_VLM_LAYERS = 32 # mode C: LM layers to keep (SmolVLA default 16 of 32)

DRIVE = '/content/drive/MyDrive/rover_vla'
DATA_TAR = f'{DRIVE}/rover_vla_v3.tar.gz'
CKPT_TAR = f'{DRIVE}/stage1_v3_pretrained.tar.gz'   # mode B
RUN_NAME = {'A': 'stage1_colab', 'B': 'stage1c_colab', 'C': 'stage1d_deeplm'}[MODE]
print('mode', MODE, '->', RUN_NAME)

## 2. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 3. Install

In [ ]:
!pip install -q "lerobot[smolvla]" wandb
import lerobot, torch
print('lerobot', lerobot.__version__, '| torch', torch.__version__)

## 4. Data (+ checkpoint for mode B)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, tarfile
os.makedirs('/content/data', exist_ok=True)
with tarfile.open(DATA_TAR) as t:
    t.extractall('/content/data')
DATASET_ROOT = '/content/data/rover_vla_v3'
print('dataset:', DATASET_ROOT, os.path.isdir(DATASET_ROOT))

POLICY_PATH = 'lerobot/smolvla_base'
if MODE == 'B':
    os.makedirs('/content/ckpt', exist_ok=True)
    with tarfile.open(CKPT_TAR) as t:
        t.extractall('/content/ckpt')
    POLICY_PATH = '/content/ckpt/pretrained_model'
    print('warm-start from:', POLICY_PATH, os.path.isdir(POLICY_PATH))
print('policy path:', POLICY_PATH)

## 5. Weights & Biases
Put your key in a Colab **secret** named `WANDB_API_KEY` (key icon, left bar).

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    import wandb; wandb.login()
print('wandb key set:', bool(os.environ.get('WANDB_API_KEY')))

## 6. Precision + mode-B patch
bf16 needs compute capability >= 8.0 (A100/L4). T4/V100 fall back to fp32.
Mode B also patches the freeze logic to unfreeze only the top-K vision layers.

In [ ]:
import torch, pathlib
cc = torch.cuda.get_device_capability()
bf16_ok = cc[0] >= 8
print('compute capability', cc, '| bf16 native:', bf16_ok)

import lerobot
f = pathlib.Path(lerobot.__file__).parent / 'policies/smolvla/smolvlm_with_expert.py'
s = f.read_text()
if not bf16_ok:
    s = s.replace('torch_dtype="bfloat16"', 'torch_dtype="float32"')
    print('patched VLM load -> float32')

if MODE == 'B':
    old = '        else:\n            # To avoid unused params issue with distributed training'
    new = ('        else:\n'
           '            for _p in self.get_vlm_model().text_model.parameters():\n'
           '                _p.requires_grad = False\n'
           '            _vm = self.get_vlm_model().vision_model\n'
           '            for _p in _vm.parameters():\n'
           '                _p.requires_grad = False\n'
           f'            for _l in _vm.encoder.layers[-{VISION_TOPK}:]:\n'
           '                for _p in _l.parameters():\n'
           '                    _p.requires_grad = True\n'
           '            # To avoid unused params issue with distributed training')
    assert old in s, 'freeze-branch anchor not found'
    s = s.replace(old, new)
    print(f'patched: top-{VISION_TOPK} vision unfreeze, LM frozen')
f.write_text(s)

gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 64 if gb > 30 else (32 if gb > 20 else 16)
if MODE == 'C':
    BATCH = max(8, BATCH // 2)   # deeper LM costs memory
print('VRAM %.0f GB -> batch %d' % (gb, BATCH))

## 7. Train
Action space: flat 10x(x,y,v)=30-dim chunk with `chunk_size=1` (lerobot's
delta-timestamps chunking would mix body frames).

In [ ]:
extra = ''
if MODE == 'B':
    extra = '--policy.train_expert_only=false --policy.freeze_vision_encoder=false'
elif MODE == 'C':
    extra = f'--policy.num_vlm_layers={NUM_VLM_LAYERS}'

OUTPUT_DIR = f'/content/outputs/{RUN_NAME}'
cmd = f'''lerobot-train \
  --policy.path={POLICY_PATH} --policy.push_to_hub=false \
  --policy.chunk_size=1 --policy.n_action_steps=1 {extra} \
  --dataset.repo_id=local/rover_vla_v3 --dataset.root={DATASET_ROOT} \
  --dataset.video_backend=pyav \
  --rename_map='{{"observation.image": "observation.images.camera1"}}' \
  --batch_size={BATCH} --steps={STEPS} --save_freq={SAVE_FREQ} --log_freq=50 \
  --wandb.enable=true --wandb.project=rover-vla --wandb.mode=online \
  --wandb.notes='{RUN_NAME} (colab, mode {MODE})' \
  --output_dir={OUTPUT_DIR}'''
print(cmd)
!{cmd}

## 8. Save the checkpoint back to Drive
Then download it locally and evaluate with `rover/runtime/policy_server.py`
+ `ros2 run rover_expert run_eval.py --swap`.

In [ ]:
import tarfile, os
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
OUT = f'{DRIVE}/checkpoints/{RUN_NAME}.tar.gz'
with tarfile.open(OUT, 'w:gz') as t:
    t.add(f'{OUTPUT_DIR}/checkpoints/last/pretrained_model', arcname='pretrained_model')
print('saved', OUT, os.path.getsize(OUT) // 1024 // 1024, 'MB')